In [1]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling
from pyproj import CRS
from matplotlib import pyplot as plt
from tqdm import tqdm
import os

In [2]:
population_2020_ = rio.open('../data/population/ppp_2020_1km_Aggregated.tif')

In [3]:
country_raster_10km_ = rio.open('../results/country-rasters/country-raster-eck4-10km.tif')
country_raster_50km_ = rio.open('../results/country-rasters/country-raster-eck4-50km.tif')

In [4]:
population_2020 = population_2020_.read(1, masked=True).filled(np.nan)

KeyboardInterrupt: 

In [ ]:
population_2020_10km = np.zeros(shape=country_raster_10km_.shape, dtype=np.float32)
population_2020_50km = np.zeros(shape=country_raster_50km_.shape, dtype=np.float32)

In [ ]:
# population_2020[population_2020 < 0]

array([], dtype=float32)

In [ ]:
reproject(
    source=population_2020,
    destination=population_2020_10km,
    src_transform=population_2020_.transform,
    src_crs=population_2020_.crs,
    dst_transform=country_raster_10km_.transform,
    dst_crs=country_raster_10km_.crs,
    resampling=Resampling.sum,
    src_nodata=np.nan,
    dst_nodata=np.nan
)

(array([[nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.]],
       shape=(1694, 3386), dtype=float32),
 Affine(10000.0, 0.0, -16930000.0,
        0.0, -10000.0, 8470000.0))

In [8]:
reproject(
    source=population_2020,
    destination=population_2020_50km,
    src_transform=population_2020_.transform,
    src_crs=population_2020_.crs,
    dst_transform=country_raster_50km_.transform,
    dst_crs=country_raster_50km_.crs,
    resampling=Resampling.sum,
    src_nodata=np.nan,
    dst_nodata=np.nan
)

(array([[nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        ...,
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.],
        [nan, nan, nan, ...,  0.,  0.,  0.]],
       shape=(340, 678), dtype=float32),
 Affine(50000.0, 0.0, -16950000.0,
        0.0, -50000.0, 8500000.0))

In [ ]:
a = np.nansum(population_2020)
b = np.nansum(population_2020_10km)
c = np.nansum(population_2020_50km)
print(a, b, c)
print((a - b) / a)
print((a - c) / a)

7.969634e+09 7.969566e+09
8.480189e-06


In [20]:
with rio.open(
    '../results/population-rasters/population-2020-10km.tif',
    "w",
    driver="GTiff",
    height=country_raster_10km_.shape[0],
    width=country_raster_10km_.shape[1],
    count=1,
    dtype=population_2020_10km.dtype,
    crs=country_raster_10km_.crs,
    transform=country_raster_10km_.transform,
    compress='lzw'
) as dst:
    dst.write(population_2020_10km, 1)

In [10]:
with rio.open(
    '../results/population-rasters/population-2020-50km.tif',
    "w",
    driver="GTiff",
    height=country_raster_50km_.shape[0],
    width=country_raster_50km_.shape[1],
    count=1,
    dtype=population_2020_50km.dtype,
    crs=country_raster_50km_.crs,
    transform=country_raster_50km_.transform,
    compress='lzw'
) as dst:
    dst.write(population_2020_50km, 1)

process future population

In [ ]:
def reproject_future_population(year, scenario='SSP2', res='10km'):
    country_raster_ = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')
    input_path = os.path.join('..', 'data', 'population', scenario, f'{scenario}_{year}.tif')
    src = rio.open(input_path)
    population_grid = src.read(1, masked=True).filled(np.nan)
    population_grid_eck4_ = np.zeros(shape=country_raster_.shape, dtype=np.float32)

    reproject(
        source=population_grid,
        destination=population_grid_eck4_,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=country_raster_.transform,
        dst_crs=country_raster_.crs,
        resampling=Resampling.sum,
        src_nodata=np.nan,
        dst_nodata=np.nan
    )

    print(np.sum(np.where(population_grid_eck4_ > 0, population_grid_eck4_, 0)) / np.nansum(population_grid))

    os.makedirs(f'../results/population-rasters/{scenario}', exist_ok=True)
    output_path = os.path.join('..', 'results', 'population-rasters', scenario, f'population-{year}-{scenario}-{res}.tif')
    with rio.open(
        output_path, 
        "w",
        driver="GTiff",
        height=country_raster_.shape[0],
        width=country_raster_.shape[1],
        count=1,
        dtype=population_grid_eck4_.dtype,
        crs=country_raster_.crs,
        transform=country_raster_.transform,
        compress='lzw'
    ) as dst:
        dst.write(population_grid_eck4_, 1)

In [ ]:
for scenario in tqdm(['SSP1', 'SSP3', 'SSP4', 'SSP5']):
    for year in tqdm([2050]):
        reproject_future_population(year, scenario=scenario, res='1km')

In [ ]:
for scenario in tqdm(['SSP2', 'SSP3', 'SSP4', 'SSP5']):
    for year in tqdm(range(2025, 2055, 5)):
        reproject_future_population(year, scenario=scenario, res='10km')

  0%|          | 0/4 [00:00<?, ?it/s]

0.9999862


0.9999859


0.9999852


0.99998474


...

0.9999852


0.9999847


0.9999841


0.9999835


100%|██████████| 4/4 [1:23:00<00:00, 1245.06s/it]


In [ ]:
for scenario in tqdm(['SSP2', 'SSP3', 'SSP4', 'SSP5']):
    for year in tqdm(range(2025, 2055, 5)):
        reproject_future_population(year, scenario=scenario, res='50km')

  0%|          | 0/1 [00:00<?, ?it/s]

0.9999901


0.99999005


0.99998975


0.99998945


0.9999894


0.99998915


100%|██████████| 1/1 [14:25<00:00, 865.82s/it]
